# 🇰🇷 Korea Revenue Forecast Pipeline
### DB 매출 데이터 → 시계열 예측 (SARIMA / ETS / Theta / Ensemble) → DB 저장
---
**기존 `universal_ts_forecast_function_v2.py` 를 수정 없이 그대로 사용합니다.**

## 0. 환경 설정 및 Import

In [5]:
import sys, os, gc, warnings, traceback
import numpy as np
import pandas as pd
from datetime import datetime
from pathlib import Path
from sqlalchemy import text

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────
# Config.py 경로 자동 탐색 (노트북 / 데스크탑 모두 지원)
# ─────────────────────────────────────────────────────────
CONFIG_PATHS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast\DATA",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA",
]

for cp in CONFIG_PATHS:
    if os.path.exists(cp) and os.path.exists(os.path.join(cp, 'config.py')):
        # DATA 폴더 추가
        if cp not in sys.path:
            sys.path.insert(0, cp)
        # DATA 상위 폴더(프로젝트 루트)도 추가 → "from DATA.xxx import" 가 작동
        parent = str(Path(cp).parent)
        if parent not in sys.path:
            sys.path.insert(0, parent)
        print(f"✅ Config 경로: {cp}")
        print(f"✅ 프로젝트 루트: {parent}")
        break

if not config_found:
    raise FileNotFoundError(
        f"config.py를 찾을 수 없습니다. 아래 경로를 확인하세요:\n" +
        "\n".join(CONFIG_PATHS)
    )

# config.py에서 DB 정보 import
from config import get_db_info, get_engine

# ─────────────────────────────────────────────────────────
# 기존 예측 함수 모듈 import (수정 없이 그대로 사용)
# 핵심 함수:
#   forecast_sarima(y, forecast_horizon, seasonal_period, ...)  → dict
#   forecast_ets(y, forecast_horizon, m, ...)                   → dict
#   forecast_theta(y, forecast_horizon, m, ...)                 → dict
#   infer_freq_alias(index)                                     → str
#   seasonal_periods_from_freq(freq_alias)                      → int
#   monitor_memory_usage(threshold_mb)                          → float
#   clear_memory()                                              → None
# ─────────────────────────────────────────────────────────
from universal_ts_forecast_function_v2 import (
    forecast_sarima,
    forecast_ets,
    forecast_theta,
    infer_freq_alias,
    seasonal_periods_from_freq,
    monitor_memory_usage,
    clear_memory,
)

print("✅ 모든 라이브러리 Import 완료")

✅ Config 경로: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
✅ 프로젝트 루트: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
✅ 모든 라이브러리 Import 완료


## 1. 전역 파라미터 설정
> ★ **이 셀만 수정하면 됩니다**

In [6]:
# ══════════════════════════════════════════════════════════
#  ★ 사용자 설정 영역
# ══════════════════════════════════════════════════════════

# DB 테이블명
SRC_TABLE  = "korea_fs_data_from_DG"          # 원천 재무 데이터
DEST_TABLE = "korea_revenue_forecast_result"   # 예측 결과 저장

# 추출할 재무 지표 (DB indicator 칼럼 값)
INDICATOR  = "매출액(천원)"

# 최소 연속 분기 수 (7년 = 28분기)
MIN_QUARTERS = 28

# 예측 분기 수 (default 8분기)
FORECAST_HORIZON = 8

# 메모리 경고 임계값 (MB)
MEMORY_THRESHOLD_MB = 2000

# ── 배치 처리 범위 ──────────────────────────────────────
# 전체 실행: None / 부분 실행: 숫자 입력
# 예) 0~499번 실행: TICKER_START=0, TICKER_END=500
TICKER_START = None
TICKER_END   = None

# 특정 티커만 실행 (None이면 전체 또는 배치 범위 실행)
# 예: SPECIFIC_TICKERS = ["000020", "005930"]
SPECIFIC_TICKERS = None

# 테스트 모드: True → 처음 N개 티커만 실행
TEST_MODE = True
TEST_N    = 2

# 배치마다 메모리 정리 간격 (티커 수)
GC_INTERVAL = 10

print("✅ 파라미터 설정 완료")
print(f"   지표       : {INDICATOR}")
print(f"   최소 분기  : {MIN_QUARTERS}  |  예측 기간: {FORECAST_HORIZON}분기")
print(f"   테스트 모드: {TEST_MODE} (N={TEST_N})")

✅ 파라미터 설정 완료
   지표       : 매출액(천원)
   최소 분기  : 28  |  예측 기간: 8분기
   테스트 모드: True (N=2)


## 2. DB 연결 확인

In [9]:
db_info = get_db_info()
engine  = get_engine(db_info)

with engine.connect() as conn:
    cnt = conn.execute(text(f"SELECT COUNT(*) FROM {SRC_TABLE}")).scalar()
    print(f"✅ DB 연결 성공 | {SRC_TABLE} 총 행 수: {cnt:,}")

✅ DB 연결 성공 | korea_fs_data_from_DG 총 행 수: 5,902,708


## 3. 대상 Ticker 추출
> 가장 최근 분기부터 `MIN_QUARTERS` 이상 데이터가 있는 종목만 선별

In [10]:
def get_valid_tickers(engine, src_table, indicator, min_quarters):
    """
    indicator 기준으로 min_quarters 이상 분기 데이터가 있는 ticker 목록 반환.
    """
    with engine.connect() as conn:
        # 가장 최근 분기 확인
        latest = conn.execute(
            text(f"SELECT MAX(date) FROM {src_table} WHERE indicator = :ind"),
            {"ind": indicator}
        ).scalar()
        print(f"   DB 최근 분기: {latest}")

        # 분기 수 충족 ticker 추출
        df = pd.read_sql(
            text(f"""
                SELECT ticker, COUNT(DISTINCT date) AS q_cnt
                FROM {src_table}
                WHERE indicator = :ind
                GROUP BY ticker
                HAVING COUNT(DISTINCT date) >= :mq
                ORDER BY ticker
            """),
            conn,
            params={"ind": indicator, "mq": min_quarters}
        )

    print(f"✅ 조건 충족 티커: {len(df):,}개 (최소 {min_quarters}분기 이상)")
    return df["ticker"].tolist()


all_tickers = get_valid_tickers(engine, SRC_TABLE, INDICATOR, MIN_QUARTERS)
print(f"   예시: {all_tickers[:5]} ...")

   DB 최근 분기: 2025-09-30
✅ 조건 충족 티커: 2,553개 (최소 28분기 이상)
   예시: ['000010', '000020', '000030', '000040', '000050'] ...


## 4. 실행 대상 Ticker 최종 결정

In [11]:
if SPECIFIC_TICKERS is not None:
    # 특정 티커 지정 모드
    ticker_list = [t for t in SPECIFIC_TICKERS if t in all_tickers]
    missing = [t for t in SPECIFIC_TICKERS if t not in all_tickers]
    print(f"[특정 티커 모드] 유효: {len(ticker_list)}개")
    if missing:
        print(f"   ⚠️  DB에 없거나 분기 부족: {missing}")
else:
    # 배치 범위 슬라이싱
    start = TICKER_START if TICKER_START is not None else 0
    end   = TICKER_END   if TICKER_END   is not None else len(all_tickers)
    ticker_list = all_tickers[start:end]
    print(f"[배치 모드] index {start} ~ {end-1}  →  {len(ticker_list)}개")

# 테스트 모드 적용
if TEST_MODE:
    ticker_list = ticker_list[:TEST_N]
    print(f"⚠️  테스트 모드: 처음 {TEST_N}개만 실행 → {ticker_list}")

print(f"\n최종 예측 대상: {len(ticker_list)}개 티커")

[배치 모드] index 0 ~ 2552  →  2553개
⚠️  테스트 모드: 처음 2개만 실행 → ['000010', '000020']

최종 예측 대상: 2개 티커


## 5. 헬퍼 함수 정의

In [12]:
# ──────────────────────────────────────────────────────────────────
# 5-1. DB에서 단일 ticker 데이터 로드 + 분기 중복 제거
# ──────────────────────────────────────────────────────────────────
def fetch_ticker_series(engine, src_table, ticker, indicator, min_quarters):
    """
    반환: pd.Series (index=DatetimeIndex, name='value')
         분기 중복 제거 완료, 시계열 오름차순
    데이터 부족 시 None 반환
    """
    sql = text(f"""
        SELECT date, value
        FROM {src_table}
        WHERE ticker    = :ticker
          AND indicator = :ind
        ORDER BY date ASC
    """)
    with engine.connect() as conn:
        df = pd.read_sql(sql, conn, params={"ticker": ticker, "ind": indicator})

    if df.empty:
        return None

    df["date"]  = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["value"]).sort_values("date")

    # 분기 중복 제거: 같은 분기(QuarterEnd) 내 마지막 날짜 1개만 유지
    # → 처음/마지막 단독 데이터도 자동 보존됨
    df["quarter"] = df["date"].dt.to_period("Q")
    df = (
        df.sort_values("date")
          .groupby("quarter", as_index=False)
          .last()          # 분기 내 최신 날짜 1개
          .sort_values("date")
          .reset_index(drop=True)
    )

    if len(df) < min_quarters:
        return None

    # DatetimeIndex pd.Series 로 반환 (기존 함수 인터페이스 맞춤)
    s = pd.Series(
        df["value"].values,
        index=pd.DatetimeIndex(df["date"]),
        name="value",
        dtype=float
    )
    return s


# ──────────────────────────────────────────────────────────────────
# 5-2. 예측 결과 dict → long-format DataFrame 변환
#      기존 함수 반환값: {"forecast": np.ndarray, "spec": {...}, ...}
# ──────────────────────────────────────────────────────────────────
def result_to_long_df(ticker, model_name, result_dict, last_date, horizon):
    """
    칼럼: date, ticker, indicator, model, value,
          forecast_date, sarima_params, created_at, updated_at
    """
    if "error" in result_dict or "forecast" not in result_dict:
        return pd.DataFrame()

    fc_arr = np.asarray(result_dict["forecast"]).flatten()[:horizon]
    spec   = result_dict.get("spec", {})

    # 예측 날짜: 마지막 실측 분기 다음 분기부터
    forecast_dates = pd.date_range(
        start  = last_date + pd.offsets.QuarterEnd(1),
        periods= horizon,
        freq   = "Q"
    )

    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    today = datetime.now().strftime("%Y-%m-%d")

    # SARIMA 파라미터 문자열화
    if model_name == "SARIMA" and spec:
        sarima_params = (
            f"order={spec.get('order')}, "
            f"seasonal_order={spec.get('seasonal_order')}, "
            f"ic_value={round(spec.get('ic_value', 0), 2)}"
        )
    else:
        sarima_params = None

    rows = []
    for fdate, fval in zip(forecast_dates, fc_arr):
        rows.append({
            "date"         : fdate.strftime("%Y-%m-%d"),
            "ticker"       : ticker,
            "indicator"    : INDICATOR,
            "model"        : model_name,
            "value"        : round(float(fval), 4) if np.isfinite(fval) else None,
            "forecast_date": today,
            "sarima_params": sarima_params,
            "created_at"   : now,
            "updated_at"   : now,
        })
    return pd.DataFrame(rows)


# ──────────────────────────────────────────────────────────────────
# 5-3. DB 중복 방지 upsert
#      키: (date, ticker, forecast_date, model) 조합이 없는 행만 INSERT
# ──────────────────────────────────────────────────────────────────
def upsert_to_db(engine, dest_table, df):
    """
    새로운 행만 INSERT. 중복 행은 기존 데이터 유지.
    반환: 신규 저장 행 수 (int)
    """
    if df.empty:
        return 0

    MERGE_KEYS = ["date", "ticker", "forecast_date", "model"]

    with engine.connect() as conn:
        try:
            existing = pd.read_sql(
                text(f"SELECT date, ticker, forecast_date, model FROM {dest_table}"),
                conn
            )
            existing["date"] = pd.to_datetime(existing["date"]).dt.strftime("%Y-%m-%d")
        except Exception:
            existing = pd.DataFrame(columns=MERGE_KEYS)

    if not existing.empty:
        merged  = df.merge(existing[MERGE_KEYS].drop_duplicates(),
                           on=MERGE_KEYS, how="left", indicator=True)
        new_df  = merged[merged["_merge"] == "left_only"].drop(columns="_merge")
    else:
        new_df = df.copy()

    if new_df.empty:
        return 0

    new_df.to_sql(
        dest_table, engine,
        if_exists="append",
        index=False,
        chunksize=500,
        method="multi"
    )
    return len(new_df)


print("✅ 헬퍼 함수 정의 완료")

✅ 헬퍼 함수 정의 완료


## 6. 단일 Ticker 예측 함수
> 기존 `forecast_sarima / forecast_ets / forecast_theta` 시그니처에 정확히 맞춤

In [13]:
def forecast_single_ticker(engine, ticker, horizon=FORECAST_HORIZON):
    """
    단일 ticker → SARIMA / ETS / Theta 예측 → Ensemble → DB 저장

    기존 함수 시그니처:
      forecast_sarima(y, forecast_horizon, seasonal_period, ...)  → dict
      forecast_ets   (y, forecast_horizon, m, ...)                → dict
      forecast_theta (y, forecast_horizon, m, ...)                → dict
    """
    # ── 데이터 로드 ──────────────────────────────────────
    y = fetch_ticker_series(engine, SRC_TABLE, ticker, INDICATOR, MIN_QUARTERS)
    if y is None:
        return 0

    # 기존 함수의 freq/seasonal 추론 사용
    freq_alias = infer_freq_alias(y.index)         # 예: "Q"
    m          = seasonal_periods_from_freq(freq_alias)  # 분기=4
    last_date  = y.index[-1]

    all_frames   = []
    fc_arrays    = {}   # Ensemble 계산용

    # ── SARIMA ───────────────────────────────────────────
    try:
        res_sarima = forecast_sarima(
            y,
            forecast_horizon = horizon,
            seasonal_period  = m,      # ← 기존 함수 파라미터명 그대로
        )
        df_s = result_to_long_df(ticker, "SARIMA", res_sarima, last_date, horizon)
        if not df_s.empty:
            all_frames.append(df_s)
            fc_arrays["SARIMA"] = np.asarray(res_sarima["forecast"]).flatten()[:horizon]
    except Exception as e:
        print(f"   ⚠️  SARIMA 실패 ({ticker}): {e}")

    # ── ETS ──────────────────────────────────────────────
    try:
        res_ets = forecast_ets(
            y,
            forecast_horizon = horizon,
            m                = m,      # ← 기존 함수 파라미터명 그대로
        )
        df_e = result_to_long_df(ticker, "ETS", res_ets, last_date, horizon)
        if not df_e.empty:
            all_frames.append(df_e)
            fc_arrays["ETS"] = np.asarray(res_ets["forecast"]).flatten()[:horizon]
    except Exception as e:
        print(f"   ⚠️  ETS 실패 ({ticker}): {e}")

    # ── Theta ─────────────────────────────────────────────
    try:
        res_theta = forecast_theta(
            y,
            forecast_horizon = horizon,
            m                = m,      # ← 기존 함수 파라미터명 그대로
        )
        df_t = result_to_long_df(ticker, "Theta", res_theta, last_date, horizon)
        if not df_t.empty:
            all_frames.append(df_t)
            fc_arrays["Theta"] = np.asarray(res_theta["forecast"]).flatten()[:horizon]
    except Exception as e:
        print(f"   ⚠️  Theta 실패 ({ticker}): {e}")

    # ── Ensemble (SARIMA + ETS + Theta 평균) ─────────────
    if len(fc_arrays) >= 2:
        try:
            ensemble_fc = np.mean(
                np.stack(list(fc_arrays.values()), axis=0), axis=0
            )
            ensemble_result = {"forecast": ensemble_fc, "spec": {}}
            df_ens = result_to_long_df(ticker, "Ensemble", ensemble_result, last_date, horizon)
            if not df_ens.empty:
                all_frames.append(df_ens)
        except Exception as e:
            print(f"   ⚠️  Ensemble 실패 ({ticker}): {e}")

    if not all_frames:
        return 0

    combined = pd.concat(all_frames, ignore_index=True)
    saved_n  = upsert_to_db(engine, DEST_TABLE, combined)

    # 예측 후 메모리 정리
    clear_memory()
    return saved_n


print("✅ 단일 Ticker 예측 함수 정의 완료")

✅ 단일 Ticker 예측 함수 정의 완료


## 7. 🧪 테스트: 1~2개 티커 단독 실행
> 전체 실행 전 이 셀로 먼저 검증하세요

In [14]:
# ════════════════════════════════════════
#  원하는 ticker 를 직접 입력해서 테스트
# ════════════════════════════════════════
TEST_TICKERS_MANUAL = ["000660","005930"]   # ← 여기만 수정

for ticker in TEST_TICKERS_MANUAL:
    print(f"\n{'='*55}")
    print(f"  테스트 예측: {ticker}")
    print(f"{'='*55}")

    # 원본 데이터 확인
    y_test = fetch_ticker_series(engine, SRC_TABLE, ticker, INDICATOR, MIN_QUARTERS)
    if y_test is None:
        print("  ❌ 데이터 부족 또는 없음 → 스킵")
        continue

    freq = infer_freq_alias(y_test.index)
    m    = seasonal_periods_from_freq(freq)
    print(f"  기간: {y_test.index[0].date()} ~ {y_test.index[-1].date()}")
    print(f"  분기 수: {len(y_test)}  |  freq={freq}  |  seasonal_period(m)={m}")
    print("  최근 5개:")
    print(y_test.tail(5).to_string())

    print(f"\n  → 예측 시작 (horizon={FORECAST_HORIZON})")
    n_saved = forecast_single_ticker(engine, ticker, horizon=FORECAST_HORIZON)
    print(f"  ✅ DB 저장 행 수: {n_saved}")

print("\n테스트 완료!")


  테스트 예측: 000660
  기간: 2004-03-31 ~ 2025-06-30
  분기 수: 86  |  freq=Q  |  seasonal_period(m)=4
  최근 5개:
date
2024-06-30    1.642326e+10
2024-09-30    1.757307e+10
2024-12-31    1.976704e+10
2025-03-31    1.763914e+10
2025-06-30    2.223195e+10

  → 예측 시작 (horizon=8)
[메모리] forecast_sarima 실행 전: 438.87 MB
[메모리] find_best_sarima_params 실행 전: 438.89 MB
[메모리] find_best_sarima_params 실행 후: 441.82 MB (변화: +2.94 MB)
[메모리] forecast_sarima 실행 후: 441.89 MB (변화: +3.02 MB)
[메모리] forecast_ets 실행 전: 441.92 MB
[메모리] forecast_ets 실행 후: 442.27 MB (변화: +0.36 MB)
[메모리] forecast_theta 실행 전: 442.27 MB
[메모리] forecast_theta 실행 후: 442.51 MB (변화: +0.23 MB)


OperationalError: (pymysql.err.OperationalError) (1054, "Unknown column 'model' in 'field list'")
[SQL: INSERT INTO korea_revenue_forecast_result (date, ticker, indicator, model, value, forecast_date, sarima_params, created_at, updated_at) VALUES (%(date_m0)s, %(ticker_m0)s, %(indicator_m0)s, %(model_m0)s, %(value_m0)s, %(forecast_date_m0)s, %(sarima_params_m0)s, %(created_at_m0)s, %(updated_at_m0)s), (%(date_m1)s, %(ticker_m1)s, %(indicator_m1)s, %(model_m1)s, %(value_m1)s, %(forecast_date_m1)s, %(sarima_params_m1)s, %(created_at_m1)s, %(updated_at_m1)s), (%(date_m2)s, %(ticker_m2)s, %(indicator_m2)s, %(model_m2)s, %(value_m2)s, %(forecast_date_m2)s, %(sarima_params_m2)s, %(created_at_m2)s, %(updated_at_m2)s), (%(date_m3)s, %(ticker_m3)s, %(indicator_m3)s, %(model_m3)s, %(value_m3)s, %(forecast_date_m3)s, %(sarima_params_m3)s, %(created_at_m3)s, %(updated_at_m3)s), (%(date_m4)s, %(ticker_m4)s, %(indicator_m4)s, %(model_m4)s, %(value_m4)s, %(forecast_date_m4)s, %(sarima_params_m4)s, %(

In [ ]:
# ── 테스트 결과 DB 조회 ──
with engine.connect() as conn:
    try:
        preview = pd.read_sql(
            text(f"""
                SELECT date, ticker, indicator, model, value,
                       forecast_date, sarima_params
                FROM {DEST_TABLE}
                WHERE ticker IN :tickers
                ORDER BY ticker, model, date
            """),
            conn,
            params={"tickers": tuple(TEST_TICKERS_MANUAL)}
        )
        print(f"저장된 결과 ({len(preview)}행):")
        display(preview)
    except Exception as e:
        print(f"❌ 조회 실패: {e}")

## 8. 전체 / 배치 실행 (메모리 관리 포함)

In [ ]:
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

total        = len(ticker_list)
success_cnt  = 0
fail_cnt     = 0
total_saved  = 0
fail_log     = []

print(f"\n{'='*60}")
print(f"  예측 시작: 총 {total}개 티커  |  예측 기간: {FORECAST_HORIZON}분기")
print(f"{'='*60}\n")

pbar = tqdm(enumerate(ticker_list), total=total, desc="예측 진행")

for i, ticker in pbar:
    pbar.set_description(f"[{i+1}/{total}] {ticker} 예측 중...")

    try:
        n_saved      = forecast_single_ticker(engine, ticker, horizon=FORECAST_HORIZON)
        total_saved += n_saved
        success_cnt += 1
    except Exception as e:
        fail_cnt += 1
        fail_log.append({"ticker": ticker, "error": str(e)})
        print(f"\n  ❌ {ticker} 오류: {e}")

    pbar.set_postfix({"저장": total_saved, "성공": success_cnt, "실패": fail_cnt})

    # ── 주기적 메모리 정리 ──
    if (i + 1) % GC_INTERVAL == 0:
        mem = monitor_memory_usage(threshold_mb=MEMORY_THRESHOLD_MB)
        pbar.write(f"   [메모리 체크] {mem:.0f} MB  (ticker {i+1}/{total})")

print(f"\n{'='*60}")
print(f"  ✅ 예측 완료")
print(f"  성공: {success_cnt}  |  실패: {fail_cnt}  |  총 저장 행: {total_saved:,}")
print(f"{'='*60}")

if fail_log:
    print("\n  ── 실패 티커 목록 ──")
    for fl in fail_log:
        print(f"    {fl['ticker']}: {fl['error']}")

## 9. 결과 확인

In [ ]:
# 전체 저장 현황 요약
with engine.connect() as conn:
    try:
        summary = pd.read_sql(
            text(f"""
                SELECT ticker, model, COUNT(*) AS rows
                FROM {DEST_TABLE}
                GROUP BY ticker, model
                ORDER BY ticker, model
            """),
            conn
        )
        print(f"티커별 모델별 저장 현황 (총 {len(summary)}행):")
        display(summary)
    except Exception as e:
        print(f"❌ 조회 실패: {e}")

In [ ]:
# 최근 저장 데이터 미리보기
with engine.connect() as conn:
    try:
        recent = pd.read_sql(
            text(f"""
                SELECT *
                FROM {DEST_TABLE}
                ORDER BY created_at DESC
                LIMIT 40
            """),
            conn
        )
        print(f"최근 저장 데이터 샘플 (40행):")
        display(recent)
    except Exception as e:
        print(f"❌ 조회 실패: {e}")